In [ ]:
import numpy as np
import soundfile as sf
import parselmouth
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

def analyze_rhythm_spectrum(audio_path, title="Rhythm Spectrum"):
    # 1. 音声読み込みと Parselmouth による音圧包絡線抽出
    snd = parselmouth.Sound(audio_path)
    # 100Hz (10ms step) で平滑化された音圧カーブを取得
    intensity = snd.to_intensity(minimum_pitch=75.0, time_step=0.01)
    env = intensity.values[0]
    t = intensity.xs()
    
    # 2. 有声区間のトリミングと正規化（無音部の影響を排除）
    threshold = np.max(env) - 30.0
    active_idx = np.where(env >= threshold)[0]
    if len(active_idx) > 0:
        env = env[active_idx[0]:active_idx[-1]]
        t = t[active_idx[0]:active_idx[-1]]
    
    # トレンド除去（DCオフセットと全体的な漸減を除去）
    env_detrend = env - np.polyval(np.polyfit(np.arange(len(env)), env, 1), np.arange(len(env)))
    # ハミング窓
    env_windowed = env_detrend * np.hamming(len(env_detrend))
    
    # 3. FFT による低周波リズムパワースペクトル計算
    sr_env = 100.0  # 10ms step = 100 Hz
    n_fft = 2048
    fft_vals = np.abs(np.fft.rfft(env_windowed, n=n_fft)) ** 2
    freqs = np.fft.rfftfreq(n_fft, d=1.0/sr_env)
    
    # プロソディ帯域 (0.5 Hz 〜 10 Hz) にフォーカス
    band_mask = (freqs >= 0.5) & (freqs <= 10.0)
    band_freqs = freqs[band_mask]
    band_power = fft_vals[band_mask]
    band_power /= np.max(band_power)  # ピーク正規化
    
    # 4. 可視化
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), gridspec_kw={'height_ratios': [1, 1.2]})
    
    # 上段: 時間領域の音圧包絡線
    ax1.plot(t - t[0], env, color="#10B981", lw=2)
    ax1.set_title(f"{title} - Intensity Envelope", fontsize=12, fontweight="bold")
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Intensity (dB)")
    ax1.grid(True, alpha=0.3)
    
    # 下段: 周波数領域のリズムスペクトル
    ax2.plot(band_freqs, band_power, color="#38BDF8", lw=2.5)
    
    # 英語ストレス帯域 (1.5 ~ 3.5 Hz) と 日本語モーラ帯域 (5.0 ~ 8.0 Hz) のハイライト
    ax2.axvspan(1.5, 3.5, color="#10B981", alpha=0.15, label="English Stress Band (1.5-3.5 Hz)")
    ax2.axvspan(5.0, 8.0, color="#EF4444", alpha=0.15, label="Japanese Mora Band (5.0-8.0 Hz)")
    
    # ピーク検出
    peaks, _ = find_peaks(band_power, height=0.3, distance=10)
    if len(peaks) > 0:
        main_peak_f = band_freqs[peaks[np.argmax(band_power[peaks])]]
        ax2.axvline(main_peak_f, color="#F59E0B", ls="--", lw=1.5, label=f"Dominant Beat: {main_peak_f:.2f} Hz")
    
    ax2.set_title("Low-Frequency Modulation Spectrum (Rhythm Frequency)", fontsize=12, fontweight="bold")
    ax2.set_xlabel("Modulation Frequency (Hz) [= Beats per second]")
    ax2.set_ylabel("Normalized Power")
    ax2.set_xlim(0.5, 10.0)
    ax2.legend(loc="upper right")
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

# テスト実行例:
fig1 = analyze_rhythm_spectrum("benchmarks/UK-Native/UK-Native_01_GeorgeWarmCa_male.wav", "UK Native")
fig2 = analyze_rhythm_spectrum("benchmarks/JP-Mora-Forced/JP-Mora-Forced_01_FrankJapanes_male.wav", "JP Mora")
plt.show()

PraatError: Cannot open file “/Users/kouichinakamura/Documents/GitHub/ninja-phonics/benchmarks/S1_stress/UK-Native/UK-Native_01_GeorgeWarmCa_male.wav”.
Sound not read from sound file “/Users/kouichinakamura/Documents/GitHub/ninja-phonics/benchmarks/S1_stress/UK-Native/UK-Native_01_GeorgeWarmCa_male.wav”.